# Real ASAS-SN fits for all July 1 VSX variable-class instances

This notebook selects **every July 1 Review-DB candidate** whose exact VSX label belongs
to one of the requested class groups, loads its real ASAS-SN light curve, and applies the
class-appropriate first model from the supplied model ladder. There is no representative
sampling limit: every conservative match is attempted and every success or failure is
persisted.

The fit families are:

| Requested class | First model implemented here |
|---|---|
| Classical/anomalous Cepheid, RRab, RRc | fixed-period harmonic Fourier ladder, order selected by profiled BIC |
| BL Her / W Vir candidate | harmonic Fourier ladder plus a seasonal stability check |
| Blazhko RR Lyrae | carrier harmonics with independently fitted seasonal amplitudes/phases |
| RRd / beat Cepheid | iterative prewhitening, simultaneous refit, and tested combination frequencies |
| $\delta$ Scuti / SX Phe | iterative multifrequency prewhitening and simultaneous refit |
| RV Tauri | compare $P$ and $2P$, then test a long secondary sinusoid |
| SRd | low-order multifrequency fit plus seasonal amplitudes/phases |
| SRa | low-order harmonic fit plus seasonal stability check |
| SRb | multifrequency fit plus seasonal amplitudes/phases |
| SRc / red supergiant | celerite2 stochastically driven damped-oscillator (SHO) Gaussian process |
| OSARG | multifrequency fit, if the July 1 DB contains a conservative VSX match |
| $\alpha$ Cyg | SHO Gaussian process, if the July 1 DB contains VSX `ACYG` |

Scientific and operational boundaries:

- VSX labels are external catalog classifications, not classifications inferred here.
- Exact configured labels are used; scientifically different classes are not substituted.
- Frequency searches reject harmonics and carrier-relative integer-day alias families before
  accepting a component.
- Fourier BIC profiles over a common multiplicative uncertainty scale, preventing underestimated
  formal errors from forcing every fit to the maximum tested order.
- One row in the saved instance table corresponds to every database match, including load or fit
  failures. Figures are saved for every successful fit but are not embedded en masse.
- The batch is a descriptive model audit, not a class-performance benchmark or physical-mode
  identification pipeline.

## Configuration and imports

Inputs are opened read-only. Results go to a new notebook-specific directory and do
not alter the Review DB, cached light curves, or prior products.

In [1]:
from __future__ import annotations

import json
import os
import sqlite3
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.timeseries import LombScargle
from celerite2 import GaussianProcess, terms
from IPython.display import display
from scipy.optimize import minimize

from malca.io.lightcurve_io import load_lightcurve_df

plt.style.use("default")
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.22})
pd.set_option("display.max_columns", 80)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "malca").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the MALCA checkout.")


REPO_ROOT = find_repo_root()
RUN_NAME = os.environ.get("MALCA_VSX_FIT_RUN", "dat3-full-extended_2026-07-01-v4")
RUN_ROOT = REPO_ROOT / "output" / "runs" / RUN_NAME
REVIEW_DB = Path(os.environ.get("MALCA_VSX_FIT_DB", RUN_ROOT / "review" / "review.db"))
OUTPUT_DIR = Path(
    os.environ.get(
        "MALCA_VSX_FIT_OUTPUT",
        REPO_ROOT / "output" / "notebooks" / "july1_vsx_class_model_ladder_all_instances",
    )
)
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MIN_POINTS = 40
WRITE_ALL_FIGURES = True
SHOW_FIGURES = False
DELTA_BIC_ACCEPT = 10.0
RANDOM_SEED = 7302026

assert REVIEW_DB.exists(), f"Missing Review DB: {REVIEW_DB}"
print(f"Repository: {REPO_ROOT}")
print(f"Read-only source: {REVIEW_DB}")
print(f"New outputs: {OUTPUT_DIR}")

Matplotlib is building the font cache; this may take a moment.


Repository: /Users/calder/code/malca
Read-only source: /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4/review/review.db
New outputs: /Users/calder/code/malca/output/notebooks/july1_vsx_class_model_ladder_all_instances


## Conservative VSX class definitions

These definitions intentionally avoid broad substring matching. For example, `RRAB`
does not absorb `RRAB/BL`, and `SDOR` is not used as a substitute for `ACYG`.
Colon-suffixed uncertain catalog types are accepted only where explicitly listed.

In [2]:
MODEL_SPECS = [
    {
        "key": "classical_cepheid",
        "label": "Classical Cepheid",
        "vsx_classes": ("DCEP-FU", "DCEP-FO", "DCEP", "CEP"),
        "family": "harmonic",
        "orders": (1, 2, 3, 4, 6, 8),
        "period_bounds": (0.5, 150.0),
    },
    {
        "key": "anomalous_cepheid",
        "label": "Anomalous Cepheid",
        "vsx_classes": ("ACEP", "ACEP:", "ACEP|CEP"),
        "family": "harmonic",
        "orders": (1, 2, 3, 4, 6),
        "period_bounds": (0.2, 10.0),
    },
    {
        "key": "type_ii_cepheid",
        "label": "BL Her / W Vir candidate",
        "vsx_classes": ("CWB", "CWA", "CW-FU", "CW-FO"),
        "family": "harmonic_stability",
        "orders": (1, 2, 3, 4, 6, 8),
        "period_bounds": (0.5, 100.0),
    },
    {
        "key": "rrab",
        "label": "RRab",
        "vsx_classes": ("RRAB",),
        "family": "harmonic",
        "orders": (1, 2, 3, 4, 6, 8),
        "period_bounds": (0.2, 1.2),
    },
    {
        "key": "rrc",
        "label": "RRc",
        "vsx_classes": ("RRC",),
        "family": "harmonic",
        "orders": (1, 2, 3, 4),
        "period_bounds": (0.15, 0.8),
    },
    {
        "key": "blazhko_rr_lyrae",
        "label": "Blazhko RR Lyrae",
        "vsx_classes": ("RRAB/BL", "RRAB/BL:"),
        "family": "time_dependent",
        "orders": (2, 3, 4, 6),
        "period_bounds": (0.2, 1.2),
    },
    {
        "key": "rrd_or_beat_cepheid",
        "label": "RRd / beat Cepheid",
        "vsx_classes": ("RRD", "DCEP-FU|DCEP-FO"),
        "family": "multifrequency_combinations",
        "max_frequencies": 2,
        "period_bounds": (0.15, 10.0),
    },
    {
        "key": "delta_scuti_sx_phe",
        "label": "delta Scuti / SX Phe",
        "vsx_classes": ("DSCT", "DSCTC", "HADS", "HADS(B)", "SXPHE"),
        "family": "multifrequency",
        "max_frequencies": 4,
        "period_bounds": (0.04, 0.5),
    },
    {
        "key": "rv_tauri",
        "label": "RV Tauri",
        "vsx_classes": ("RVA", "RVB", "RV"),
        "family": "double_period_long",
        "orders": (1, 2, 3, 4, 6),
        "period_bounds": (10.0, 300.0),
    },
    {
        "key": "srd",
        "label": "SRd",
        "vsx_classes": ("SRD",),
        "family": "multifrequency_time",
        "max_frequencies": 3,
        "period_bounds": (5.0, 500.0),
    },
    {
        "key": "sra",
        "label": "SRa",
        "vsx_classes": ("SRA",),
        "family": "harmonic_stability",
        "orders": (1, 2, 3),
        "period_bounds": (10.0, 1000.0),
    },
    {
        "key": "srb",
        "label": "SRb",
        "vsx_classes": ("SRB", "SRB:"),
        "family": "multifrequency_time",
        "max_frequencies": 3,
        "period_bounds": (10.0, 1200.0),
    },
    {
        "key": "src_red_supergiant",
        "label": "SRc / red supergiant",
        "vsx_classes": ("SRC",),
        "family": "stochastic_sho",
        "period_bounds": (10.0, 2000.0),
    },
    {
        "key": "osarg",
        "label": "OSARG",
        "vsx_classes": ("OSARG",),
        "family": "multifrequency",
        "max_frequencies": 4,
        "period_bounds": (2.0, 300.0),
    },
    {
        "key": "alpha_cyg",
        "label": "alpha Cyg / yellow hypergiant",
        "vsx_classes": ("ACYG",),
        "family": "stochastic_sho",
        "period_bounds": (1.0, 2000.0),
    },
]

REQUIRED_COLUMNS = [
    "candidate_id", "asas_sn_id", "lc_path", "ra", "dec",
    "vsx_class", "vsx_period", "vsx_sep_arcsec",
    "periodicity_period", "period_for_fold_days",
    "stats_variability_lomb_scargle_best_period_days",
    "stats_n_unique_nights", "stats_time_span_days",
]

with sqlite3.connect(f"file:{REVIEW_DB}?mode=ro", uri=True) as conn:
    existing_columns = {
        row[1] for row in conn.execute("PRAGMA table_info(candidates)").fetchall()
    }
    selected_columns = [column for column in REQUIRED_COLUMNS if column in existing_columns]
    candidates = pd.read_sql_query(
        f"SELECT {', '.join(selected_columns)} FROM candidates",
        conn,
    )

for column in REQUIRED_COLUMNS:
    if column not in candidates:
        candidates[column] = np.nan

candidates["vsx_class"] = candidates["vsx_class"].fillna("").astype(str).str.strip().str.upper()
for column in [
    "vsx_period", "vsx_sep_arcsec", "periodicity_period", "period_for_fold_days",
    "stats_variability_lomb_scargle_best_period_days",
    "stats_n_unique_nights", "stats_time_span_days",
]:
    candidates[column] = pd.to_numeric(candidates[column], errors="coerce")

availability_rows = []
for spec in MODEL_SPECS:
    matched = candidates["vsx_class"].isin(spec["vsx_classes"])
    availability_rows.append(
        {
            "class_key": spec["key"],
            "requested_class": spec["label"],
            "model_family": spec["family"],
            "accepted_vsx_labels": "|".join(spec["vsx_classes"]),
            "n_vsx_matches": int(matched.sum()),
            "n_with_vsx_period": int((matched & candidates["vsx_period"].gt(0)).sum()),
            "n_with_lc_path": int(
                (matched & candidates["lc_path"].fillna("").astype(str).str.strip().ne("")).sum()
            ),
        }
    )

availability = pd.DataFrame(availability_rows)
availability.to_csv(OUTPUT_DIR / "availability_by_vsx_group.csv", index=False)
display(availability)

,class_key,requested_class,model_family,accepted_vsx_labels,n_vsx_matches,n_with_vsx_period,n_with_lc_path
0,classical_cepheid,Classical Cepheid,harmonic,DCEP-FU|DCEP-FO|DCEP|CEP,10,10,10
1,anomalous_cepheid,Anomalous Cepheid,harmonic,ACEP|ACEP:|ACEP|CEP,5,5,5
2,type_ii_cepheid,BL Her / W Vir candidate,harmonic_stability,CWB|CWA|CW-FU|CW-FO,8,8,8
3,rrab,RRab,harmonic,RRAB,162,162,162
4,rrc,RRc,harmonic,RRC,25,25,25
5,blazhko_rr_lyrae,Blazhko RR Lyrae,time_dependent,RRAB/BL|RRAB/BL:,21,21,21
6,rrd_or_beat_cepheid,RRd / beat Cepheid,multifrequency_combinations,RRD|DCEP-FU|DCEP-FO,2,2,2
7,delta_scuti_sx_phe,delta Scuti / SX Phe,multifrequency,DSCT|DSCTC|HADS|HADS(B)|SXPHE,18,17,18
8,rv_tauri,RV Tauri,double_period_long,RVA|RVB|RV,4,4,4
9,srd,SRd,multifrequency_time,SRD,5,5,5


## Load every real light curve

Every exact VSX match in each configured group is attempted. The selected photometric
band is the one with the most good measurements. Linear models retain all cameras in
that band and fit camera offsets as nuisance terms. The SHO GP uses the single camera
maximizing sampling and time span so the stochastic kernel is not asked to absorb
camera zero-point steps.

In [3]:
def resolve_lc_path(row: pd.Series) -> Path | None:
    raw = str(row.get("lc_path", "") or "").strip()
    probes = []
    if raw:
        probes.extend([Path(raw), RUN_ROOT / raw, REPO_ROOT / raw])
        probes.append(RUN_ROOT / "bundle_assets" / "lightcurves" / Path(raw).name)
    asas_id = str(row.get("asas_sn_id", "") or "").strip()
    if asas_id:
        probes.extend(
            [
                RUN_ROOT / "bundle_assets" / "lightcurves" / f"{asas_id}.dat3",
                RUN_ROOT / "bundle_assets" / "lightcurves" / f"{asas_id}.parquet",
            ]
        )
    for probe in probes:
        if probe.exists() and probe.is_file():
            return probe.resolve()
    return None


def prepare_lightcurve(path: Path) -> pd.DataFrame:
    frame = load_lightcurve_df(path, apply_quality=True).copy()
    for column in ("jd", "mag", "mag_err"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.loc[np.isfinite(frame["jd"]) & np.isfinite(frame["mag"])].copy()
    if frame.empty:
        return frame

    frame["band"] = frame["band"].fillna("unknown").astype(str)
    best_band = frame["band"].value_counts().sort_values(ascending=False).index[0]
    frame = frame.loc[frame["band"].eq(best_band)].copy()
    frame["camera"] = frame["camera"].fillna("unknown").astype(str)

    positive_error = frame["mag_err"].where(frame["mag_err"].gt(0))
    fallback = float(positive_error.median())
    if not np.isfinite(fallback) or fallback <= 0:
        scatter = 1.4826 * np.nanmedian(np.abs(frame["mag"] - np.nanmedian(frame["mag"])))
        fallback = max(float(scatter) * 0.1, 0.02)
    floor = max(
        float(positive_error.quantile(0.05)) if positive_error.notna().any() else fallback,
        0.003,
    )
    frame["mag_err"] = positive_error.fillna(fallback).clip(lower=floor)
    return frame.sort_values("jd").reset_index(drop=True)


def initial_period_from_row(row: pd.Series) -> tuple[float, str]:
    for column, source in [
        ("vsx_period", "VSX"),
        ("period_for_fold_days", "MALCA period_for_fold"),
        ("periodicity_period", "MALCA periodicity"),
        ("stats_variability_lomb_scargle_best_period_days", "MALCA Lomb-Scargle"),
    ]:
        value = pd.to_numeric(pd.Series([row.get(column)]), errors="coerce").iloc[0]
        if np.isfinite(value) and value > 0:
            return float(value), source
    return np.nan, "none"


instance_rows = []
lightcurves: dict[str, pd.DataFrame] = {}

for spec in MODEL_SPECS:
    subset = candidates.loc[candidates["vsx_class"].isin(spec["vsx_classes"])].copy()
    subset = subset.sort_values(["vsx_class", "candidate_id"], kind="stable")
    for _, row in subset.iterrows():
        candidate_id = str(row["candidate_id"])
        fit_id = f"{spec['key']}__{candidate_id}"
        path = resolve_lc_path(row)
        period, period_source = initial_period_from_row(row)
        record = {
            "fit_id": fit_id,
            "class_key": spec["key"],
            "requested_class": spec["label"],
            "model_family": spec["family"],
            "candidate_id": candidate_id,
            "asas_sn_id": row["asas_sn_id"],
            "vsx_class": row["vsx_class"],
            "vsx_period_days": row["vsx_period"],
            "period_used_initial_days": period,
            "period_source": period_source,
            "vsx_sep_arcsec": row["vsx_sep_arcsec"],
            "input_status": "pending",
            "input_note": "",
        }
        if path is None:
            record.update(input_status="missing_lightcurve_path", input_note="No resolved local file.")
            instance_rows.append(record)
            continue
        try:
            lc = prepare_lightcurve(path)
        except Exception as exc:
            record.update(
                input_status=f"load_failed_{type(exc).__name__}",
                input_note=str(exc),
                lc_path=str(path),
            )
            instance_rows.append(record)
            continue
        if len(lc) < MIN_POINTS:
            record.update(
                input_status="insufficient_same_band_points",
                input_note=f"{len(lc)} good points; minimum is {MIN_POINTS}.",
                lc_path=str(path),
                band=(lc["band"].iloc[0] if not lc.empty else ""),
                n_points=len(lc),
            )
            instance_rows.append(record)
            continue

        lightcurves[fit_id] = lc
        record.update(
            input_status="ready",
            lc_path=str(path),
            band=lc["band"].iloc[0],
            n_points=len(lc),
            n_cameras=lc["camera"].nunique(),
            time_span_days=float(lc["jd"].max() - lc["jd"].min()),
        )
        instance_rows.append(record)

instances = pd.DataFrame(instance_rows)
instances.to_csv(OUTPUT_DIR / "all_instances.csv", index=False)

instance_coverage = (
    instances.groupby(["class_key", "requested_class"], dropna=False)
    .agg(
        n_db_instances=("candidate_id", "size"),
        n_ready=("input_status", lambda values: int(values.eq("ready").sum())),
        n_input_failures=("input_status", lambda values: int(values.ne("ready").sum())),
    )
    .reset_index()
)
zero_groups = availability.loc[availability["n_vsx_matches"].eq(0), ["class_key", "requested_class"]].copy()
if not zero_groups.empty:
    zero_groups["n_db_instances"] = 0
    zero_groups["n_ready"] = 0
    zero_groups["n_input_failures"] = 0
    instance_coverage = pd.concat([instance_coverage, zero_groups], ignore_index=True)
instance_coverage = instance_coverage.sort_values("class_key")
instance_coverage.to_csv(OUTPUT_DIR / "instance_coverage.csv", index=False)
display(instance_coverage)
print(
    f"Loaded {int(instances['input_status'].eq('ready').sum())} / {len(instances)} "
    "conservative VSX-matched light curves."
)

,class_key,requested_class,n_db_instances,n_ready,n_input_failures
14,alpha_cyg,alpha Cyg / yellow hypergiant,0,0,0
0,anomalous_cepheid,Anomalous Cepheid,5,5,0
1,blazhko_rr_lyrae,Blazhko RR Lyrae,21,21,0
2,classical_cepheid,Classical Cepheid,10,10,0
3,delta_scuti_sx_phe,delta Scuti / SX Phe,18,18,0
13,osarg,OSARG,0,0,0
4,rrab,RRab,162,162,0
5,rrc,RRc,25,25,0
6,rrd_or_beat_cepheid,RRd / beat Cepheid,2,2,0
7,rv_tauri,RV Tauri,4,4,0


Loaded 309 / 309 conservative VSX-matched light curves.


## Weighted Fourier, alias-safe prewhitening, seasonal, and SHO helpers

Fourier coefficients use simultaneous weighted least squares. Model-order BIC profiles
over a common multiplicative uncertainty scale:

\[
\mathrm{BIC}_{\rm profiled}
  = n\log(\chi^2/n) + (k+1)\log n,
\]

where the extra parameter is the fitted global error scale. This preserves relative
measurement weights without allowing underestimated formal errors to force ever-higher
Fourier orders. Prewhitening rejects integer harmonics through order 10 and frequencies
separated from an accepted component by an integer number of cycles/day.

In [4]:
@dataclass
class LinearFit:
    coefficients: np.ndarray
    prediction: np.ndarray
    residual: np.ndarray
    chi2: float
    bic: float
    reduced_chi2: float
    wrmse: float
    error_scale_factor: float
    n_parameters: int
    camera_levels: tuple[str, ...]
    components: tuple[tuple[str, float, int], ...]


def camera_columns(camera: pd.Series) -> tuple[np.ndarray, tuple[str, ...]]:
    values = camera.fillna("unknown").astype(str).to_numpy()
    levels = tuple(sorted(pd.unique(values)))
    columns = [np.ones(len(values), dtype=float)]
    for level in levels[1:]:
        columns.append((values == level).astype(float))
    return np.column_stack(columns), levels


def design_matrix(
    lc: pd.DataFrame,
    components: list[tuple[str, float, int]],
    *,
    t0: float | None = None,
) -> tuple[np.ndarray, tuple[str, ...]]:
    base, camera_levels = camera_columns(lc["camera"])
    t = lc["jd"].to_numpy(float)
    t0 = float(np.nanmin(t)) if t0 is None else float(t0)
    columns = [base]
    for _, frequency, harmonics in components:
        for harmonic in range(1, int(harmonics) + 1):
            angle = 2.0 * np.pi * harmonic * frequency * (t - t0)
            columns.extend([np.cos(angle)[:, None], np.sin(angle)[:, None]])
    return np.column_stack(columns), camera_levels


def weighted_linear_fit(
    lc: pd.DataFrame,
    components: list[tuple[str, float, int]],
    *,
    t0: float | None = None,
) -> LinearFit:
    X, camera_levels = design_matrix(lc, components, t0=t0)
    y = lc["mag"].to_numpy(float)
    dy = lc["mag_err"].to_numpy(float)
    weight_sqrt = 1.0 / dy
    Xw = X * weight_sqrt[:, None]
    yw = y * weight_sqrt
    coefficients, _, _, _ = np.linalg.lstsq(Xw, yw, rcond=None)
    prediction = X @ coefficients
    residual = y - prediction
    chi2 = float(np.sum(np.square(residual / dy)))
    n, k = X.shape
    profiled_scale2 = max(chi2 / max(n, 1), np.finfo(float).tiny)
    bic = float(n * np.log(profiled_scale2) + (k + 1) * np.log(n))
    error_scale_factor = float(np.sqrt(chi2 / max(n - k, 1)))
    return LinearFit(
        coefficients=coefficients,
        prediction=prediction,
        residual=residual,
        chi2=chi2,
        bic=bic,
        reduced_chi2=float(chi2 / (n - k)) if n > k else np.nan,
        wrmse=float(np.sqrt(np.average(np.square(residual), weights=1.0 / np.square(dy)))),
        error_scale_factor=error_scale_factor,
        n_parameters=k,
        camera_levels=camera_levels,
        components=tuple(components),
    )


def camera_corrected_magnitude(lc: pd.DataFrame, fit: LinearFit) -> np.ndarray:
    camera = lc["camera"].fillna("unknown").astype(str).to_numpy()
    offsets = np.zeros(len(lc), dtype=float)
    for index, level in enumerate(fit.camera_levels[1:], start=1):
        offsets[camera == level] = fit.coefficients[index]
    return lc["mag"].to_numpy(float) - offsets


def signal_prediction(
    t: np.ndarray,
    fit: LinearFit,
    *,
    t0: float,
) -> np.ndarray:
    n_camera_parameters = len(fit.camera_levels)
    output = np.full(len(t), fit.coefficients[0], dtype=float)
    coefficient_index = n_camera_parameters
    for _, frequency, harmonics in fit.components:
        for harmonic in range(1, harmonics + 1):
            angle = 2.0 * np.pi * harmonic * frequency * (t - t0)
            output += (
                fit.coefficients[coefficient_index] * np.cos(angle)
                + fit.coefficients[coefficient_index + 1] * np.sin(angle)
            )
            coefficient_index += 2
    return output


def carrier_component_prediction(
    t: np.ndarray,
    fit: LinearFit,
    *,
    t0: float,
) -> np.ndarray:
    output = np.full(len(t), fit.coefficients[0], dtype=float)
    if not fit.components:
        return output
    coefficient_index = len(fit.camera_levels)
    _, frequency, harmonics = fit.components[0]
    for harmonic in range(1, harmonics + 1):
        angle = 2.0 * np.pi * harmonic * frequency * (t - t0)
        output += (
            fit.coefficients[coefficient_index] * np.cos(angle)
            + fit.coefficients[coefficient_index + 1] * np.sin(angle)
        )
        coefficient_index += 2
    return output


def component_amplitude_phase(
    fit: LinearFit,
    component_index: int = 0,
    harmonic: int = 1,
) -> tuple[float, float]:
    coefficient_index = len(fit.camera_levels)
    for index, (_, _, n_harmonics) in enumerate(fit.components):
        if index == component_index:
            coefficient_index += 2 * (harmonic - 1)
            a_cos, b_sin = fit.coefficients[coefficient_index : coefficient_index + 2]
            return float(np.hypot(a_cos, b_sin)), float(np.arctan2(-b_sin, a_cos))
        coefficient_index += 2 * n_harmonics
    return np.nan, np.nan


def harmonic_ladder(
    lc: pd.DataFrame,
    period: float,
    orders: tuple[int, ...],
) -> dict[str, Any]:
    rows, fits = [], {}
    for order in orders:
        fit = weighted_linear_fit(lc, [("carrier", 1.0 / period, int(order))])
        fits[int(order)] = fit
        rows.append(
            {
                "harmonic_order": int(order), "bic": fit.bic,
                "reduced_chi2": fit.reduced_chi2, "wrmse": fit.wrmse,
            }
        )
    table = pd.DataFrame(rows).sort_values("harmonic_order")
    best_order = int(table.loc[table["bic"].idxmin(), "harmonic_order"])
    return {"fit": fits[best_order], "best_order": best_order, "ladder": table}


def frequency_grid(t: np.ndarray, min_period: float, max_period: float) -> np.ndarray:
    baseline = float(np.ptp(t))
    minimum_frequency = 1.0 / max_period
    maximum_frequency = 1.0 / min_period
    n_grid = int(np.clip(4.0 * baseline * (maximum_frequency - minimum_frequency), 4_000, 60_000))
    return np.linspace(minimum_frequency, maximum_frequency, n_grid)


def carrier_relative_daily_alias(
    frequency: float,
    existing: list[float],
    baseline: float,
) -> tuple[bool, float, int]:
    tolerance = max(3.0 / baseline, 0.02)
    best_distance = np.inf
    best_order = 0
    for other in existing:
        delta = frequency - other
        alias_order = int(np.rint(delta))
        if alias_order == 0:
            continue
        distance = abs(delta - alias_order)
        if distance < best_distance:
            best_distance = distance
            best_order = alias_order
    return bool(best_distance < tolerance), float(best_distance), int(best_order)


def is_independent_frequency(frequency: float, existing: list[float], baseline: float) -> bool:
    resolution = max(3.0 / baseline, 1e-6)
    if absolute_daily_alias_distance(frequency) < max(resolution, 0.02):
        return False
    harmonic_ratios = tuple(1.0 / order for order in range(2, 11)) + tuple(
        float(order) for order in range(1, 11)
    )
    for other in existing:
        for ratio in harmonic_ratios:
            comparison = ratio * other
            separation = abs(frequency - comparison)
            relative_separation = separation / max(abs(frequency), abs(comparison), 1e-12)
            if separation < resolution or relative_separation < 0.01:
                return False
    is_alias, _, _ = carrier_relative_daily_alias(frequency, existing, baseline)
    return not is_alias


def strongest_independent_frequency(
    lc: pd.DataFrame,
    residual: np.ndarray,
    min_period: float,
    max_period: float,
    existing: list[float],
    *,
    reject_absolute_daily_aliases: bool = False,
) -> tuple[float, float]:
    t = lc["jd"].to_numpy(float)
    dy = lc["mag_err"].to_numpy(float)
    frequencies = frequency_grid(t, min_period, max_period)
    power = LombScargle(t, residual, dy, fit_mean=True, center_data=True).power(frequencies)
    order = np.argsort(power)[::-1]
    baseline = float(np.ptp(t))
    absolute_alias_tolerance = max(3.0 / baseline, 0.02)
    for index in order[: min(500, len(order))]:
        frequency = float(frequencies[index])
        if (
            reject_absolute_daily_aliases
            and absolute_daily_alias_distance(frequency) < absolute_alias_tolerance
        ):
            continue
        if is_independent_frequency(frequency, existing, baseline):
            return frequency, float(power[index])
    return np.nan, np.nan


def absolute_daily_alias_distance(frequency: float) -> float:
    return float(abs(frequency - np.round(frequency)))


def fit_multifrequency(
    lc: pd.DataFrame,
    *,
    initial_period: float,
    period_bounds: tuple[float, float],
    max_frequencies: int,
    allow_combinations: bool = False,
) -> dict[str, Any]:
    min_period, max_period = period_bounds
    max_period = min(float(max_period), max(float(np.ptp(lc["jd"])) / 2.0, min_period * 1.01))
    frequencies: list[float] = []
    components: list[tuple[str, float, int]] = []
    rows = []
    base_fit = weighted_linear_fit(lc, [])
    current = base_fit

    if np.isfinite(initial_period) and min_period <= initial_period <= max_period:
        frequencies.append(1.0 / initial_period)
        components.append(("catalog_carrier", frequencies[-1], 1))
        current = weighted_linear_fit(lc, components)
        rows.append(
            {
                "component": "catalog_carrier",
                "frequency_per_day": frequencies[-1],
                "period_days": initial_period,
                "lomb_scargle_power": np.nan,
                "delta_bic": base_fit.bic - current.bic,
                "absolute_daily_alias_distance_per_day": absolute_daily_alias_distance(frequencies[-1]),
                "accepted": True,
            }
        )
    while len(frequencies) < int(max_frequencies):
        frequency, power = strongest_independent_frequency(
            lc, current.residual, min_period, max_period, frequencies
        )
        if not np.isfinite(frequency):
            break
        is_alias, alias_distance, alias_order = carrier_relative_daily_alias(
            frequency, frequencies, float(np.ptp(lc["jd"]))
        )
        candidate_components = components + [(f"independent_{len(frequencies) + 1}", frequency, 1)]
        candidate = weighted_linear_fit(lc, candidate_components)
        delta_bic = current.bic - candidate.bic
        rows.append(
            {
                "component": f"independent_{len(frequencies) + 1}",
                "frequency_per_day": frequency,
                "period_days": 1.0 / frequency,
                "lomb_scargle_power": power,
                "delta_bic": delta_bic,
                "absolute_daily_alias_distance_per_day": absolute_daily_alias_distance(frequency),
                "carrier_relative_daily_alias_distance_per_day": alias_distance,
                "carrier_relative_daily_alias_order": alias_order,
                "accepted": bool(delta_bic >= DELTA_BIC_ACCEPT and not is_alias),
            }
        )
        if delta_bic < DELTA_BIC_ACCEPT or is_alias:
            break
        frequencies.append(frequency)
        components = candidate_components
        current = candidate

    if allow_combinations and len(frequencies) >= 2:
        f1, f2 = frequencies[:2]
        combination_candidates = [
            ("f1+f2", f1 + f2),
            ("abs(f1-f2)", abs(f1 - f2)),
            ("2f1+f2", 2.0 * f1 + f2),
            ("f1+2f2", f1 + 2.0 * f2),
        ]
        fmin, fmax = 1.0 / max_period, 1.0 / min_period
        for name, frequency in combination_candidates:
            if not (fmin <= frequency <= fmax):
                continue
            candidate_components = components + [(name, frequency, 1)]
            candidate = weighted_linear_fit(lc, candidate_components)
            delta_bic = current.bic - candidate.bic
            rows.append(
                {
                    "component": name,
                    "frequency_per_day": frequency,
                    "period_days": 1.0 / frequency,
                    "lomb_scargle_power": np.nan,
                    "delta_bic": delta_bic,
                    "absolute_daily_alias_distance_per_day": absolute_daily_alias_distance(frequency),
                    "accepted": bool(delta_bic >= DELTA_BIC_ACCEPT),
                }
            )
            if delta_bic >= DELTA_BIC_ACCEPT:
                components = candidate_components
                current = candidate

    accepted_table = pd.DataFrame(rows)
    return {"fit": current, "frequency_table": accepted_table}


def fit_seasonal_components(
    lc: pd.DataFrame,
    components: list[tuple[str, float, int]],
    global_fit: LinearFit,
) -> dict[str, Any]:
    frame = lc.copy()
    frame["_season"] = np.floor((frame["jd"] - frame["jd"].min()) / 365.25).astype(int)
    prediction = global_fit.prediction.copy()
    rows = []
    total_chi2 = 0.0
    total_parameters = 0
    n_modeled = 0

    minimum = max(25, 2 * sum(component[2] for component in components) + 8)
    reference_t0 = float(lc["jd"].min())
    longest_period = max(1.0 / component[1] for component in components)
    robust_range = float(lc["mag"].quantile(0.95) - lc["mag"].quantile(0.05))
    maximum_amplitude = max(3.0 * robust_range, 0.2)
    for season, subset in frame.groupby("_season", sort=True):
        season_span = float(np.ptp(subset["jd"]))
        if len(subset) < minimum or season_span < max(20.0, 0.75 * longest_period):
            continue
        design, _ = design_matrix(subset, components, t0=reference_t0)
        weighted_design = design / subset["mag_err"].to_numpy(float)[:, None]
        if not np.isfinite(weighted_design).all() or np.linalg.cond(weighted_design) > 1e8:
            continue
        seasonal_fit = weighted_linear_fit(subset, components, t0=reference_t0)
        amplitude, phase = component_amplitude_phase(seasonal_fit, 0, 1)
        modeled_range = float(np.ptp(seasonal_fit.prediction))
        if (
            not np.isfinite(amplitude)
            or amplitude > maximum_amplitude
            or modeled_range > max(5.0 * robust_range, 0.5)
        ):
            continue
        positions = subset.index.to_numpy()
        prediction[positions] = seasonal_fit.prediction
        total_chi2 += seasonal_fit.chi2
        total_parameters += seasonal_fit.n_parameters
        n_modeled += len(subset)
        rows.append(
            {
                "season": int(season),
                "mid_jd": float(np.median(subset["jd"])),
                "n_points": len(subset),
                "carrier_amplitude_mag": amplitude,
                "carrier_phase_rad": phase,
                "wrmse": seasonal_fit.wrmse,
            }
        )

    season_table = pd.DataFrame(rows)
    if n_modeled:
        unmodeled = np.ones(len(frame), dtype=bool)
        for season in season_table.get("season", pd.Series(dtype=int)):
            unmodeled &= frame["_season"].to_numpy() != season
        total_chi2 += float(
            np.sum(np.square((lc["mag"].to_numpy()[unmodeled] - prediction[unmodeled]) / lc["mag_err"].to_numpy()[unmodeled]))
        )
        if unmodeled.any():
            total_parameters += global_fit.n_parameters
        seasonal_bic = (
            len(lc) * np.log(max(total_chi2 / len(lc), np.finfo(float).tiny))
            + (total_parameters + 1) * np.log(len(lc))
        )
    else:
        seasonal_bic = np.nan
    return {
        "prediction": prediction,
        "season_table": season_table,
        "seasonal_bic": float(seasonal_bic) if np.isfinite(seasonal_bic) else np.nan,
        "delta_bic_vs_global": (
            float(global_fit.bic - seasonal_bic) if np.isfinite(seasonal_bic) else np.nan
        ),
    }


def fit_rv_tauri(
    lc: pd.DataFrame,
    initial_period: float,
    orders: tuple[int, ...],
) -> dict[str, Any]:
    candidates_to_test = []
    for multiplier in (1.0, 2.0):
        period = initial_period * multiplier
        ladder = harmonic_ladder(lc, period, orders)
        candidates_to_test.append((period, multiplier, ladder))
    formal_period, multiplier, best = min(candidates_to_test, key=lambda item: item[2]["fit"].bic)
    current = best["fit"]

    baseline = float(np.ptp(lc["jd"]))
    long_min = max(5.0 * formal_period, 50.0)
    long_max = baseline / 2.0
    long_frequency = np.nan
    long_delta_bic = np.nan
    accepted = False
    if long_max > long_min * 1.05:
        long_frequency, _ = strongest_independent_frequency(
            lc, current.residual, long_min, long_max, [1.0 / formal_period]
        )
        if np.isfinite(long_frequency):
            candidate_components = list(current.components) + [("long_secondary", long_frequency, 1)]
            candidate = weighted_linear_fit(lc, candidate_components)
            long_delta_bic = current.bic - candidate.bic
            if long_delta_bic >= DELTA_BIC_ACCEPT:
                current = candidate
                accepted = True
    return {
        "fit": current,
        "best_order": best["best_order"],
        "ladder": best["ladder"],
        "formal_period_days": formal_period,
        "catalog_period_multiplier": multiplier,
        "long_period_days": (1.0 / long_frequency if np.isfinite(long_frequency) else np.nan),
        "long_delta_bic": long_delta_bic,
        "long_component_accepted": accepted,
    }


def fit_sho_gp(lc: pd.DataFrame, period_hint: float = np.nan) -> dict[str, Any]:
    camera_summary = (
        lc.groupby("camera")
        .agg(n=("jd", "size"), span=("jd", lambda values: float(np.ptp(values))))
        .assign(score=lambda frame: frame["n"] * np.log1p(frame["span"]))
        .sort_values(["score", "n"], ascending=False)
    )
    chosen_camera = str(camera_summary.index[0])
    gp_lc = lc.loc[lc["camera"].astype(str).eq(chosen_camera)].copy()
    if len(gp_lc) < MIN_POINTS:
        gp_lc = lc.copy()
        medians = gp_lc.groupby("camera")["mag"].transform("median")
        gp_lc["mag"] = gp_lc["mag"] - medians + gp_lc["mag"].median()
        chosen_camera = "camera-median-corrected-all"

    gp_lc = gp_lc.sort_values("jd").reset_index(drop=True)
    t = gp_lc["jd"].to_numpy(float)
    t_rel = t - t.min()
    y = gp_lc["mag"].to_numpy(float)
    dy = gp_lc["mag_err"].to_numpy(float)
    baseline = float(np.ptp(t_rel))
    scatter = max(float(np.std(y)), 0.01)
    cadence = max(float(np.median(np.diff(np.unique(t_rel)))), 0.05)
    median_error = max(float(np.median(dy)), 0.003)
    rho_guesses = [max(baseline / 8.0, cadence * 5)]
    if np.isfinite(period_hint) and period_hint > cadence:
        rho_guesses.insert(0, min(period_hint, baseline))

    bounds = [
        (np.log(scatter * 0.02), np.log(scatter * 10.0)),
        (np.log(cadence), np.log(max(baseline * 2.0, cadence * 2))),
        (np.log(0.05), np.log(100.0)),
        (np.log(max(median_error * 0.05, 1e-5)), np.log(max(scatter * 3.0, median_error))),
        (float(np.min(y) - scatter), float(np.max(y) + scatter)),
    ]

    def evaluate(theta: np.ndarray, return_gp: bool = False):
        sigma, rho, quality, jitter = np.exp(theta[:4])
        mean = float(theta[4])
        kernel = terms.SHOTerm(sigma=sigma, rho=rho, Q=quality)
        gp = GaussianProcess(kernel, mean=mean)
        try:
            gp.compute(t_rel, yerr=np.sqrt(np.square(dy) + jitter**2))
            nll = -float(gp.log_likelihood(y))
        except Exception:
            return (1e100, None) if return_gp else 1e100
        return (nll, gp) if return_gp else nll

    starts = []
    for rho in rho_guesses:
        for quality in (0.3, 2.0, 10.0):
            starts.append(
                np.array(
                    [
                        np.log(scatter), np.log(np.clip(rho, cadence, baseline * 2)),
                        np.log(quality), np.log(median_error), np.median(y),
                    ],
                    dtype=float,
                )
            )
    solutions = [minimize(evaluate, start, method="L-BFGS-B", bounds=bounds) for start in starts]
    solution = min(solutions, key=lambda item: float(item.fun))
    nll, gp = evaluate(solution.x, return_gp=True)
    if gp is None or not np.isfinite(nll):
        raise RuntimeError("SHO GP optimization failed.")

    sigma, rho, quality, jitter = np.exp(solution.x[:4])
    grid_rel = np.linspace(0, baseline, 800)
    prediction, variance = gp.predict(y, t=grid_rel, return_var=True)
    data_prediction = gp.predict(y, t=t_rel, return_cov=False)
    residual = y - data_prediction
    bic = 2.0 * nll + 5 * np.log(len(y))
    return {
        "lc": gp_lc,
        "camera": chosen_camera,
        "sigma_mag": float(sigma),
        "rho_days": float(rho),
        "omega0_rad_per_day": float(2.0 * np.pi / rho),
        "quality_factor_q": float(quality),
        "s0": float(sigma**2 / ((2.0 * np.pi / rho) * quality)),
        "jitter_mag": float(jitter),
        "mean_mag": float(solution.x[4]),
        "nll": float(nll),
        "bic": float(bic),
        "grid_jd": grid_rel + t.min(),
        "prediction": prediction,
        "prediction_std": np.sqrt(np.maximum(variance, 0)),
        "data_prediction": data_prediction,
        "residual": residual,
        "wrmse": float(np.sqrt(np.average(np.square(residual), weights=1.0 / np.square(dy)))),
        "optimizer_success": bool(solution.success),
        "optimizer_message": str(solution.message),
    }

## Execute real fits for every available instance

All `ready` rows are fitted. If a periodic class lacks a usable catalog/MALCA period,
the notebook obtains a bounded Lomb–Scargle carrier and records that fallback. Outputs
include the selected fit, every linear coefficient, every tested harmonic order,
accepted/rejected frequency components, seasonal measurements, and failures.

In [5]:
def ensure_period_for_spec(
    lc: pd.DataFrame,
    period: float,
    period_source: str,
    spec: dict[str, Any],
) -> tuple[float, str]:
    minimum_period, maximum_period = spec["period_bounds"]
    if np.isfinite(period) and minimum_period <= period <= maximum_period:
        return float(period), period_source
    base_fit = weighted_linear_fit(lc, [])
    frequency, _ = strongest_independent_frequency(
        lc,
        base_fit.residual,
        minimum_period,
        min(maximum_period, max(float(np.ptp(lc["jd"])) / 2.0, minimum_period * 1.01)),
        [],
        reject_absolute_daily_aliases=True,
    )
    if not np.isfinite(frequency) or frequency <= 0:
        raise RuntimeError("No usable carrier period from catalog fields or bounded Lomb-Scargle.")
    return float(1.0 / frequency), "notebook bounded Lomb-Scargle fallback"


def linear_parameter_records(
    fit_id: str,
    selected: dict[str, Any],
    fit: LinearFit,
) -> list[dict[str, Any]]:
    common = {
        "fit_id": fit_id,
        "class_key": selected["class_key"],
        "candidate_id": selected["candidate_id"],
        "vsx_class": selected["vsx_class"],
    }
    rows = [
        {
            **common,
            "parameter_kind": "intercept",
            "camera": fit.camera_levels[0],
            "component": "",
            "frequency_per_day": np.nan,
            "harmonic": 0,
            "coefficient_axis": "constant",
            "value": float(fit.coefficients[0]),
        }
    ]
    for index, camera in enumerate(fit.camera_levels[1:], start=1):
        rows.append(
            {
                **common,
                "parameter_kind": "camera_offset",
                "camera": camera,
                "component": "",
                "frequency_per_day": np.nan,
                "harmonic": 0,
                "coefficient_axis": "offset_from_reference",
                "value": float(fit.coefficients[index]),
            }
        )
    coefficient_index = len(fit.camera_levels)
    for component, frequency, harmonics in fit.components:
        for harmonic in range(1, harmonics + 1):
            cos_value = float(fit.coefficients[coefficient_index])
            sin_value = float(fit.coefficients[coefficient_index + 1])
            amplitude = float(np.hypot(cos_value, sin_value))
            phase = float(np.arctan2(-sin_value, cos_value))
            for axis, value in [
                ("cos", cos_value),
                ("sin", sin_value),
                ("amplitude", amplitude),
                ("cosine_phase_rad", phase),
            ]:
                rows.append(
                    {
                        **common,
                        "parameter_kind": "signal",
                        "camera": "",
                        "component": component,
                        "frequency_per_day": frequency,
                        "harmonic": harmonic,
                        "coefficient_axis": axis,
                        "value": value,
                    }
                )
            coefficient_index += 2
    return rows


fit_results: dict[str, dict[str, Any]] = {}
fit_summary_rows = []
frequency_rows = []
seasonal_rows = []
parameter_rows = []
harmonic_order_rows = []

ready_instances = instances.loc[instances["input_status"].eq("ready")].copy()
spec_lookup = {spec["key"]: spec for spec in MODEL_SPECS}
total_ready = len(ready_instances)

for progress, (_, instance) in enumerate(ready_instances.iterrows(), start=1):
    selected = instance.to_dict()
    fit_id = selected["fit_id"]
    key = selected["class_key"]
    spec = spec_lookup[key]
    family = spec["family"]
    lc = lightcurves[fit_id]
    initial_period = float(selected["period_used_initial_days"])
    fit_period_source = str(selected["period_source"])
    result: dict[str, Any] = {
        "family": family,
        "lc": lc,
        "selected": selected,
        "fit_id": fit_id,
    }

    try:
        if family != "stochastic_sho":
            initial_period, fit_period_source = ensure_period_for_spec(
                lc, initial_period, fit_period_source, spec
            )
            result["selected"]["period_used_initial_days"] = initial_period
            result["selected"]["fit_period_source"] = fit_period_source

        if family == "harmonic":
            result.update(harmonic_ladder(lc, initial_period, spec["orders"]))
        elif family in {"harmonic_stability", "time_dependent"}:
            fitted = harmonic_ladder(lc, initial_period, spec["orders"])
            result.update(fitted)
            result.update(
                fit_seasonal_components(lc, list(fitted["fit"].components), fitted["fit"])
            )
        elif family in {"multifrequency", "multifrequency_combinations", "multifrequency_time"}:
            fitted = fit_multifrequency(
                lc,
                initial_period=initial_period,
                period_bounds=spec["period_bounds"],
                max_frequencies=spec["max_frequencies"],
                allow_combinations=(family == "multifrequency_combinations"),
            )
            result.update(fitted)
            if family == "multifrequency_time":
                result.update(
                    fit_seasonal_components(
                        lc, list(fitted["fit"].components), fitted["fit"]
                    )
                )
        elif family == "double_period_long":
            result.update(fit_rv_tauri(lc, initial_period, spec["orders"]))
        elif family == "stochastic_sho":
            result["gp"] = fit_sho_gp(lc, period_hint=initial_period)
            result["lc"] = result["gp"]["lc"]
        else:
            raise ValueError(f"Unknown family: {family}")

        result["fit_status"] = "ok"
        fit_results[fit_id] = result

        common_summary = {
            "fit_id": fit_id,
            "class_key": key,
            "requested_class": spec["label"],
            "fit_status": "ok",
            "candidate_id": selected["candidate_id"],
            "asas_sn_id": selected["asas_sn_id"],
            "vsx_class": selected["vsx_class"],
            "vsx_period_days": selected["vsx_period_days"],
            "period_input_days": initial_period,
            "period_input_source": fit_period_source,
            "model_family": family,
            "n_points_fit": len(result["lc"]),
        }
        if family == "stochastic_sho":
            gp = result["gp"]
            summary = {
                **common_summary,
                "bic": gp["bic"],
                "wrmse_mag": gp["wrmse"],
                "sho_rho_days": gp["rho_days"],
                "sho_q": gp["quality_factor_q"],
                "sho_s0": gp["s0"],
                "sho_sigma_mag": gp["sigma_mag"],
                "sho_jitter_mag": gp["jitter_mag"],
                "fit_note": f"SHO GP using camera selection: {gp['camera']}",
            }
            for parameter_name in [
                "sigma_mag", "rho_days", "omega0_rad_per_day", "quality_factor_q",
                "s0", "jitter_mag", "mean_mag",
            ]:
                parameter_rows.append(
                    {
                        "fit_id": fit_id,
                        "class_key": key,
                        "candidate_id": selected["candidate_id"],
                        "vsx_class": selected["vsx_class"],
                        "parameter_kind": "sho_gp",
                        "camera": gp["camera"],
                        "component": "sho",
                        "frequency_per_day": np.nan,
                        "harmonic": 0,
                        "coefficient_axis": parameter_name,
                        "value": gp[parameter_name],
                    }
                )
        else:
            fit = result["fit"]
            carrier_period = 1.0 / fit.components[0][1] if fit.components else np.nan
            summary = {
                **common_summary,
                "bic": fit.bic,
                "formal_reduced_chi2": fit.reduced_chi2,
                "profiled_error_scale_factor": fit.error_scale_factor,
                "wrmse_mag": fit.wrmse,
                "carrier_period_days": carrier_period,
                "harmonic_order": result.get(
                    "best_order", fit.components[0][2] if fit.components else np.nan
                ),
                "n_fitted_components": len(fit.components),
                "seasonal_delta_bic": result.get("delta_bic_vs_global", np.nan),
                "fit_note": (
                    "Seasonal delta BIC is a flexibility diagnostic, not proof of modulation."
                    if "season_table" in result else ""
                ),
            }
            if family == "double_period_long":
                summary.update(
                    {
                        "carrier_period_days": result["formal_period_days"],
                        "catalog_period_multiplier": result["catalog_period_multiplier"],
                        "long_period_days": result["long_period_days"],
                        "long_component_delta_bic": result["long_delta_bic"],
                        "long_component_accepted": result["long_component_accepted"],
                    }
                )
            parameter_rows.extend(linear_parameter_records(fit_id, result["selected"], fit))

        fit_summary_rows.append(summary)

        frequency_table = result.get("frequency_table")
        if isinstance(frequency_table, pd.DataFrame) and not frequency_table.empty:
            table = frequency_table.copy()
            table.insert(0, "vsx_class", selected["vsx_class"])
            table.insert(0, "candidate_id", selected["candidate_id"])
            table.insert(0, "class_key", key)
            table.insert(0, "fit_id", fit_id)
            frequency_rows.extend(table.to_dict("records"))

        season_table = result.get("season_table")
        if isinstance(season_table, pd.DataFrame) and not season_table.empty:
            table = season_table.copy()
            table.insert(0, "vsx_class", selected["vsx_class"])
            table.insert(0, "candidate_id", selected["candidate_id"])
            table.insert(0, "class_key", key)
            table.insert(0, "fit_id", fit_id)
            seasonal_rows.extend(table.to_dict("records"))

        ladder = result.get("ladder")
        if isinstance(ladder, pd.DataFrame) and not ladder.empty:
            table = ladder.copy()
            table.insert(0, "vsx_class", selected["vsx_class"])
            table.insert(0, "candidate_id", selected["candidate_id"])
            table.insert(0, "class_key", key)
            table.insert(0, "fit_id", fit_id)
            harmonic_order_rows.extend(table.to_dict("records"))
    except Exception as exc:
        fit_summary_rows.append(
            {
                "fit_id": fit_id,
                "class_key": key,
                "requested_class": spec["label"],
                "fit_status": f"failed_{type(exc).__name__}",
                "candidate_id": selected["candidate_id"],
                "asas_sn_id": selected["asas_sn_id"],
                "vsx_class": selected["vsx_class"],
                "vsx_period_days": selected["vsx_period_days"],
                "period_input_days": initial_period,
                "period_input_source": fit_period_source,
                "model_family": family,
                "fit_note": str(exc),
            }
        )
        warnings.warn(f"{fit_id}: {type(exc).__name__}: {exc}")

    if progress == 1 or progress % 10 == 0 or progress == total_ready:
        print(f"Fit progress: {progress}/{total_ready}")

fit_summary = pd.DataFrame(fit_summary_rows)
frequency_components = pd.DataFrame(frequency_rows)
seasonal_components = pd.DataFrame(seasonal_rows)
model_parameters = pd.DataFrame(parameter_rows)
harmonic_order_scores = pd.DataFrame(harmonic_order_rows)

fit_summary.to_csv(OUTPUT_DIR / "fit_summary_all_instances.csv", index=False)
frequency_components.to_csv(OUTPUT_DIR / "frequency_components_all_instances.csv", index=False)
seasonal_components.to_csv(OUTPUT_DIR / "seasonal_components_all_instances.csv", index=False)
model_parameters.to_csv(OUTPUT_DIR / "model_parameters_all_instances.csv", index=False)
harmonic_order_scores.to_csv(OUTPUT_DIR / "harmonic_order_scores_all_instances.csv", index=False)

class_fit_coverage = (
    instances.groupby(["class_key", "requested_class"], dropna=False)
    .agg(
        n_db_instances=("candidate_id", "size"),
        n_ready=("input_status", lambda values: int(values.eq("ready").sum())),
    )
    .reset_index()
)
fit_counts = (
    fit_summary.groupby("class_key")
    .agg(
        n_fit_attempted=("fit_id", "size"),
        n_fit_ok=("fit_status", lambda values: int(values.eq("ok").sum())),
        n_fit_failed=("fit_status", lambda values: int(values.ne("ok").sum())),
    )
    .reset_index()
)
class_fit_coverage = class_fit_coverage.merge(fit_counts, on="class_key", how="left")
class_fit_coverage = availability[
    ["class_key", "requested_class", "n_vsx_matches"]
].merge(class_fit_coverage, on=["class_key", "requested_class"], how="left")
for column in ["n_db_instances", "n_ready", "n_fit_attempted", "n_fit_ok", "n_fit_failed"]:
    class_fit_coverage[column] = class_fit_coverage[column].fillna(0).astype(int)
class_fit_coverage.to_csv(OUTPUT_DIR / "class_fit_coverage.csv", index=False)
display(class_fit_coverage)
display(fit_summary.head(20))

Fit progress: 1/309
Fit progress: 10/309


Fit progress: 20/309


Fit progress: 30/309
Fit progress: 40/309


Fit progress: 50/309


Fit progress: 60/309
Fit progress: 70/309


...

Fit progress: 260/309
Fit progress: 270/309
Fit progress: 280/309


Fit progress: 290/309
Fit progress: 300/309


Fit progress: 309/309


,class_key,requested_class,n_vsx_matches,n_db_instances,n_ready,n_fit_attempted,n_fit_ok,n_fit_failed
0,classical_cepheid,Classical Cepheid,10,10,10,10,10,0
1,anomalous_cepheid,Anomalous Cepheid,5,5,5,5,5,0
2,type_ii_cepheid,BL Her / W Vir candidate,8,8,8,8,8,0
3,rrab,RRab,162,162,162,162,162,0
4,rrc,RRc,25,25,25,25,25,0
5,blazhko_rr_lyrae,Blazhko RR Lyrae,21,21,21,21,21,0
6,rrd_or_beat_cepheid,RRd / beat Cepheid,2,2,2,2,2,0
7,delta_scuti_sx_phe,delta Scuti / SX Phe,18,18,18,18,18,0
8,rv_tauri,RV Tauri,4,4,4,4,4,0
9,srd,SRd,5,5,5,5,5,0


,fit_id,class_key,requested_class,fit_status,candidate_id,asas_sn_id,vsx_class,vsx_period_days,period_input_days,period_input_source,model_family,n_points_fit,bic,formal_reduced_chi2,profiled_error_scale_factor,wrmse_mag,carrier_period_days,harmonic_order,n_fitted_components,seasonal_delta_bic,fit_note,catalog_period_multiplier,long_period_days,long_component_delta_bic,long_component_accepted,sho_rho_days,sho_q,sho_s0,sho_sigma_mag,sho_jitter_mag
0,classical_cepheid__stv_188979638405,classical_cepheid,Classical Cepheid,ok,stv_188979638405,188979638405,CEP,19.608040,19.608040,VSX,harmonic,1480,4866.871268,25.558015,5.055494,0.052831,19.608040,1.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,classical_cepheid__stv_300648551623,classical_cepheid,Classical Cepheid,ok,stv_300648551623,300648551623,CEP,4.520195,4.520195,VSX,harmonic,1130,1486.612231,3.549121,1.883911,0.018863,4.520195,1.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,classical_cepheid__stv_592706455013,classical_cepheid,Classical Cepheid,ok,stv_592706455013,592706455013,CEP,3.623200,3.623200,VSX,harmonic,3907,14840.853826,43.883015,6.624426,0.081800,3.623200,1.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,classical_cepheid__stv_68719842566,classical_cepheid,Classical Cepheid,ok,stv_68719842566,68719842566,CEP,3.204562,3.204562,VSX,harmonic,1631,6674.013950,57.072430,7.554630,0.075331,3.204562,2.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,classical_cepheid__stv_120260151432,classical_cepheid,Classical Cepheid,ok,stv_120260151432,120260151432,DCEP,21.752570,21.752570,VSX,harmonic,1020,4749.881950,94.755805,9.734259,0.119934,21.752570,6.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,classical_cepheid__stv_146029753729,classical_cepheid,Classical Cepheid,ok,stv_146029753729,146029753729,DCEP,11.600000,11.600000,VSX,harmonic,1467,10111.177338,938.786587,30.639624,0.315599,11.600000,1.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,classical_cepheid__stv_25770624777,classical_cepheid,Classical Cepheid,ok,stv_25770624777,25770624777,DCEP,11.823440,11.823440,VSX,harmonic,706,2060.033660,16.812173,4.100265,0.040728,11.823440,3.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,classical_cepheid__stv_8590738371,classical_cepheid,Classical Cepheid,ok,stv_8590738371,8590738371,DCEP,5.259980,5.259980,VSX,harmonic,611,1167.231922,5.602192,2.366895,0.023296,5.259980,8.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,classical_cepheid__stv_369368512531,classical_cepheid,Classical Cepheid,ok,stv_369368512531,369368512531,DCEP-FU,5.036300,5.036300,VSX,harmonic,2154,8192.496520,43.330025,6.582555,0.065995,5.036300,2.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,classical_cepheid__stv_635655727916,classical_cepheid,Classical Cepheid,ok,stv_635655727916,635655727916,DCEP-FU,33.100000,33.100000,VSX,harmonic,2831,9218.512637,25.252598,5.025196,0.050259,33.100000,2.0,1.0,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Per-instance diagnostic figures

A full diagnostic PNG is saved for every successful fit under one directory per class.
The notebook does not embed hundreds of figures by default; set `SHOW_FIGURES=True` if
interactive display is desired.

In [6]:
def plot_fit(result: dict[str, Any], *, show: bool = False) -> Path:
    selected = result["selected"]
    key = selected["class_key"]
    candidate_id = selected["candidate_id"]
    label = selected["requested_class"]
    family = result["family"]
    lc = result["lc"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), constrained_layout=True)
    ax_raw, ax_model, ax_resid, ax_diag = axes.ravel()

    for camera, subset in lc.groupby("camera", sort=True):
        ax_raw.scatter(subset["jd"], subset["mag"], s=8, alpha=0.55, label=str(camera))
    ax_raw.set(
        title=f"{label}: {candidate_id} ({selected['vsx_class']})",
        xlabel="JD",
        ylabel=f"{selected['band']} magnitude",
    )
    ax_raw.invert_yaxis()
    if lc["camera"].nunique() <= 8:
        ax_raw.legend(fontsize=6, ncol=2)

    if family == "stochastic_sho":
        gp = result["gp"]
        ax_model.scatter(lc["jd"], lc["mag"], s=8, alpha=0.4, color="0.3")
        ax_model.plot(gp["grid_jd"], gp["prediction"], color="#d62728", lw=1.5)
        ax_model.fill_between(
            gp["grid_jd"],
            gp["prediction"] - gp["prediction_std"],
            gp["prediction"] + gp["prediction_std"],
            color="#d62728",
            alpha=0.18,
        )
        ax_model.set(
            title="SHO GP posterior mean and 1-sigma",
            xlabel="JD",
            ylabel="magnitude",
        )
        ax_model.invert_yaxis()
        residual = gp["residual"]
        residual_time = lc["jd"].to_numpy()
        ax_diag.bar(
            ["rho (d)", "Q", "sigma (mag)"],
            [gp["rho_days"], gp["quality_factor_q"], gp["sigma_mag"]],
            color=["#4c78a8", "#f58518", "#54a24b"],
        )
        ax_diag.set_yscale("log")
        ax_diag.set_title("Fitted SHO parameters (log scale)")
    else:
        fit = result["fit"]
        corrected = camera_corrected_magnitude(lc, fit)
        t = lc["jd"].to_numpy(float)
        carrier_period = 1.0 / fit.components[0][1] if fit.components else np.nan
        if np.isfinite(carrier_period):
            phase = np.mod((t - t.min()) / carrier_period, 1.0)
            grid_phase = np.linspace(0, 1, 600)
            grid_t = t.min() + grid_phase * carrier_period
            if len(fit.components) > 1:
                model_grid = carrier_component_prediction(grid_t, fit, t0=t.min())
                fold_title = f"Carrier-only fold: P={carrier_period:.6g} d"
            else:
                model_grid = signal_prediction(grid_t, fit, t0=t.min())
                fold_title = f"Carrier fold: P={carrier_period:.6g} d"
            ax_model.scatter(phase, corrected, s=9, alpha=0.5, color="0.25")
            ax_model.scatter(phase + 1, corrected, s=9, alpha=0.22, color="0.25")
            ax_model.plot(grid_phase, model_grid, color="#d62728", lw=1.7)
            ax_model.plot(grid_phase + 1, model_grid, color="#d62728", lw=1.7)
            ax_model.set(
                title=fold_title,
                xlabel="phase",
                ylabel="camera-corrected magnitude",
            )
            ax_model.invert_yaxis()
        else:
            order = np.argsort(t)
            ax_model.scatter(t, corrected, s=8, alpha=0.45)
            ax_model.plot(
                t[order],
                signal_prediction(t[order], fit, t0=t.min()),
                color="#d62728",
            )

        residual = (
            lc["mag"].to_numpy() - result["prediction"]
            if "prediction" in result
            else fit.residual
        )
        residual_time = t
        season_table = result.get("season_table")
        if isinstance(season_table, pd.DataFrame) and not season_table.empty:
            ax_diag.plot(
                season_table["mid_jd"],
                season_table["carrier_amplitude_mag"],
                marker="o",
                color="#4c78a8",
            )
            ax_diag.set(
                title="Time-dependent carrier amplitude",
                xlabel="season mid-JD",
                ylabel="amplitude (mag)",
            )
        elif isinstance(result.get("frequency_table"), pd.DataFrame) and not result["frequency_table"].empty:
            table = result["frequency_table"]
            relative_alias = (
                table["carrier_relative_daily_alias_order"].fillna(0).ne(0)
                & table["carrier_relative_daily_alias_distance_per_day"].lt(0.02)
            )
            colors = np.where(
                table["accepted"] & relative_alias,
                "#f58518",
                np.where(table["accepted"], "#54a24b", "#e45756"),
            )
            ax_diag.barh(table["component"], table["period_days"], color=colors)
            ax_diag.set_xscale("log")
            ax_diag.set(
                title="Tested periods (green accepted; red rejected)",
                xlabel="period (days)",
            )
        elif family == "double_period_long":
            text = (
                f"VSX multiplier: {result['catalog_period_multiplier']:.0f}x\n"
                f"formal period: {result['formal_period_days']:.5g} d\n"
                f"long period: {result['long_period_days']:.5g} d\n"
                f"long accepted: {result['long_component_accepted']}\n"
                f"Delta BIC long: {result['long_delta_bic']:.2f}"
            )
            ax_diag.text(
                0.05, 0.95, text, transform=ax_diag.transAxes, va="top", family="monospace"
            )
            ax_diag.set_axis_off()
        else:
            ladder = result.get("ladder")
            ax_diag.plot(
                ladder["harmonic_order"], ladder["bic"], marker="o", color="#4c78a8"
            )
            ax_diag.axvline(result["best_order"], color="#d62728", ls="--")
            ax_diag.set(
                title="Harmonic order selected by profiled BIC",
                xlabel="K",
                ylabel="profiled BIC",
            )

    ax_resid.scatter(residual_time, residual, s=8, alpha=0.5, color="#4c78a8")
    ax_resid.axhline(0, color="0.2", lw=0.8)
    ax_resid.set(title="Fit residuals", xlabel="JD", ylabel="observed - model (mag)")
    fig.suptitle(f"{label} | {family}", fontsize=14)
    class_directory = FIGURE_DIR / key
    class_directory.mkdir(parents=True, exist_ok=True)
    output = class_directory / f"{candidate_id}.png"
    fig.savefig(output, dpi=140, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    return output


figure_rows = []
if WRITE_ALL_FIGURES:
    total_figures = len(fit_results)
    for progress, result in enumerate(fit_results.values(), start=1):
        path = plot_fit(result, show=SHOW_FIGURES)
        selected = result["selected"]
        figure_rows.append(
            {
                "fit_id": result["fit_id"],
                "class_key": selected["class_key"],
                "candidate_id": selected["candidate_id"],
                "vsx_class": selected["vsx_class"],
                "figure_path": str(path),
            }
        )
        if progress == 1 or progress % 25 == 0 or progress == total_figures:
            print(f"Figure progress: {progress}/{total_figures}")

figure_manifest = pd.DataFrame(figure_rows)
figure_manifest.to_csv(OUTPUT_DIR / "figure_manifest_all_instances.csv", index=False)
display(figure_manifest.head(20))

Figure progress: 1/309


Figure progress: 25/309


Figure progress: 50/309


Figure progress: 75/309


Figure progress: 100/309


...

Figure progress: 250/309


Figure progress: 275/309


Figure progress: 300/309


Figure progress: 309/309


,fit_id,class_key,candidate_id,vsx_class,figure_path
0,classical_cepheid__stv_188979638405,classical_cepheid,stv_188979638405,CEP,/Users/calder/code/malca/output/notebooks/july...
1,classical_cepheid__stv_300648551623,classical_cepheid,stv_300648551623,CEP,/Users/calder/code/malca/output/notebooks/july...
2,classical_cepheid__stv_592706455013,classical_cepheid,stv_592706455013,CEP,/Users/calder/code/malca/output/notebooks/july...
3,classical_cepheid__stv_68719842566,classical_cepheid,stv_68719842566,CEP,/Users/calder/code/malca/output/notebooks/july...
4,classical_cepheid__stv_120260151432,classical_cepheid,stv_120260151432,DCEP,/Users/calder/code/malca/output/notebooks/july...
5,classical_cepheid__stv_146029753729,classical_cepheid,stv_146029753729,DCEP,/Users/calder/code/malca/output/notebooks/july...
6,classical_cepheid__stv_25770624777,classical_cepheid,stv_25770624777,DCEP,/Users/calder/code/malca/output/notebooks/july...
7,classical_cepheid__stv_8590738371,classical_cepheid,stv_8590738371,DCEP,/Users/calder/code/malca/output/notebooks/july...
8,classical_cepheid__stv_369368512531,classical_cepheid,stv_369368512531,DCEP-FU,/Users/calder/code/malca/output/notebooks/july...
9,classical_cepheid__stv_635655727916,classical_cepheid,stv_635655727916,DCEP-FU,/Users/calder/code/malca/output/notebooks/july...


## Machine-readable run metadata and interpretation guardrails

The final metadata records exact database-instance coverage, fit success, and artifact
paths. Re-run after the Review DB or bundled light curves change.

In [7]:
n_db_instances = int(len(instances))
n_ready = int(instances["input_status"].eq("ready").sum())
n_successful_fits = int(fit_summary["fit_status"].eq("ok").sum())
n_failed_fits = int(fit_summary["fit_status"].ne("ok").sum())
run_metadata = {
    "review_db": str(REVIEW_DB.resolve()),
    "run_name": RUN_NAME,
    "selection_contract": (
        "Every candidate whose exact VSX class is in a configured requested group; "
        "no representative limit and no scientifically different substitutions."
    ),
    "model_selection": (
        "Weighted least squares with a profiled multiplicative uncertainty-scale BIC."
    ),
    "frequency_guard": (
        "Reject integer harmonics through order 10, absolute integer-day window peaks, "
        "and carrier-relative integer-day aliases."
    ),
    "delta_bic_accept": DELTA_BIC_ACCEPT,
    "minimum_same_band_points": MIN_POINTS,
    "n_requested_groups": len(MODEL_SPECS),
    "n_database_instances": n_db_instances,
    "n_ready_lightcurves": n_ready,
    "n_input_failures": int(n_db_instances - n_ready),
    "n_successful_fits": n_successful_fits,
    "n_failed_fits": n_failed_fits,
    "n_figures": int(len(figure_manifest)),
    "zero_match_groups": availability.loc[
        availability["n_vsx_matches"].eq(0), "class_key"
    ].tolist(),
    "interpretation": [
        "VSX labels are catalog evidence, not classifications produced by these fits.",
        "Profiled BIC selects descriptive model complexity; it does not identify physical modes.",
        "Frequency components remain candidates even after harmonic and daily-alias guards.",
        "Seasonal models have more flexibility; their delta BIC is a modulation diagnostic.",
        "Per-class performance requires population-level held-out evaluation beyond this batch.",
    ],
    "artifacts": {
        "availability": str(OUTPUT_DIR / "availability_by_vsx_group.csv"),
        "all_instances": str(OUTPUT_DIR / "all_instances.csv"),
        "instance_coverage": str(OUTPUT_DIR / "instance_coverage.csv"),
        "class_fit_coverage": str(OUTPUT_DIR / "class_fit_coverage.csv"),
        "fit_summary": str(OUTPUT_DIR / "fit_summary_all_instances.csv"),
        "model_parameters": str(OUTPUT_DIR / "model_parameters_all_instances.csv"),
        "harmonic_order_scores": str(
            OUTPUT_DIR / "harmonic_order_scores_all_instances.csv"
        ),
        "frequencies": str(OUTPUT_DIR / "frequency_components_all_instances.csv"),
        "seasonal": str(OUTPUT_DIR / "seasonal_components_all_instances.csv"),
        "figure_manifest": str(OUTPUT_DIR / "figure_manifest_all_instances.csv"),
        "figures": str(FIGURE_DIR),
    },
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
display(pd.Series(run_metadata).to_frame("value"))
if n_successful_fits != n_ready:
    warnings.warn(
        f"Batch incomplete: {n_successful_fits}/{n_ready} ready light curves fitted successfully."
    )
print(
    f"Completed {n_successful_fits}/{n_db_instances} database-instance fits; "
    f"{len(figure_manifest)} figures written."
)

,value
review_db,/Users/calder/code/malca/output/runs/dat3-full...
run_name,dat3-full-extended_2026-07-01-v4
selection_contract,Every candidate whose exact VSX class is in a ...
model_selection,Weighted least squares with a profiled multipl...
frequency_guard,Reject integer harmonics through order 10 and ...
delta_bic_accept,10.0
minimum_same_band_points,40
n_requested_groups,15
n_database_instances,309
n_ready_lightcurves,309


Completed 309/309 database-instance fits; 309 figures written.
